# 03 - Casos de cierre desde splits reales

Este notebook usa solo `vsr_models/splits/*.csv`: no carga ROIs, videos, checkpoints ni GPU. Sirve para mostrar como transformar texto real del proyecto en casos livianos de evaluacion de cierre.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "realtime").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

## Generar casos desde `val.csv`

La utilidad crea prefijos etiquetados como `wait` y frases completas suficientemente largas como `commit`. Es un benchmark liviano, no ground truth humano final.

In [ ]:
from realtime.src.dataset_cierre import build_cases_from_split_csv

split_path = ROOT / "vsr_models" / "splits" / "val.csv"
cases = build_cases_from_split_csv(split_path, limit=12)
len(cases), cases[:3]

## Evaluar la baseline sobre esos casos

Esto permite medir rapido si una heuristica o LLM cambia commits prematuros, waits innecesarios y latencia.

In [ ]:
import json
from realtime.src.evaluar_cierre import evaluate_closure
from realtime.src.provider_factory import make_closure_provider

summary = evaluate_closure(cases, make_closure_provider("heuristic"))
compact = {k: summary[k] for k in [
    "count", "accuracy", "commit_precision", "commit_recall",
    "premature_commit_rate", "unnecessary_wait_rate", "low_confidence_recall",
    "fallbacks", "latency_ms"
]}
print(json.dumps(compact, indent=2, ensure_ascii=False))

In [ ]:
for row in summary["rows"][:8]:
    print(row["case"], "expected=", row["expected"], "predicted=", row["predicted"], "reason=", row["reason"])

## Siguiente paso natural

Cuando haya un provider Ollama/Qwen corriendo, repetir la misma evaluacion con `make_closure_provider("ollama")` y comparar contra la heuristica sin cambiar el dataset.